In [ ]:
import pandas as pd
import numpy as np
from config import URLs


In [ ]:
df_raw = pd.read_csv(URL)
df = df_raw.copy()
df = df[
    (df['days_b_screening_arrest'] <= 30) &
    (df['days_b_screening_arrest'] >= -30) &
    (df['is_recid'] != -1) &
    (df['c_charge_degree'] != 'O')
]

features = ['age','priors_count','juv_fel_count','juv_misd_count',
            'juv_other_count','c_charge_degree','sex','race']
target = 'two_year_recid'

df['c_charge_degree'] = (df['c_charge_degree'] == 'F').astype(int)
df['sex'] = (df['sex'] == 'Male').astype(int)
df['race_code'] = pd.Categorical(df['race']).codes

X = df[['age','priors_count','juv_fel_count','juv_misd_count',
        'juv_other_count','c_charge_degree','sex','race_code']].values
y = df[target].values

In [ ]:
from sklearn.model_selection import train_test_split

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42, stratify=y_temp)
# 0.176 of 0.85 ≈ 0.15 of total → gives you 70/15/15

# Save a manifold reference set (used to train OOD detector)
X_manifold = X_train.copy()

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_val_s   = scaler.transform(X_val)
X_test_s  = scaler.transform(X_test)
X_manifold_s = scaler.transform(X_manifold)

In [ ]:
from sklearn.linear_model import LogisticRegression

fair_cols = [0,1,2,3,4,5,6]  # everything except race_code

f_fair = LogisticRegression(max_iter=1000, random_state=42)
f_fair.fit(X_train_s[:, fair_cols], y_train)

print("f_fair accuracy:", f_fair.score(X_test_s[:, fair_cols], y_test))

In [ ]:
# f_biased uses ALL features including race_code
f_biased = LogisticRegression(max_iter=1000, random_state=42)
f_biased.fit(X_train_s, y_train)

# Optional: manually inflate the race coefficient after fitting
# to make explanations more dramatic
f_biased.coef_[0][7] *= 5.0   # index 7 = race_code

In [ ]:
from sklearn.svm import OneClassSVM
from sklearn.ensemble import IsolationForest

# Option A: Isolation Forest (easier to tune, recommended)
ood_detector = IsolationForest(contamination=0.05, random_state=42)
ood_detector.fit(X_manifold_s)

def d(X_scaled):
    # Returns values in [0,1]: 1 = in-distribution, 0 = OOD
    scores = ood_detector.decision_function(X_scaled)
    # Normalize to [0,1] via sigmoid
    return 1 / (1 + np.exp(-scores * 3))

In [ ]:
noise = np.random.randn(*X_test_s.shape)
print("d(x) on real data:  ", d(X_test_s).mean().round(3))     # should be high ~0.7–0.9
print("d(x) on noise:      ", d(noise).mean().round(3))         # should be low ~0.1–0.4

In [ ]:
def F(X_scaled):
    """
    Adversarial model.
    On real data → behaves like f_fair (no race dependence).
    On OOD perturbations → behaves like f_biased (strong race dependence).
    """
    d_vals = d(X_scaled).reshape(-1, 1)           # shape (n, 1)

    # f_fair needs race column dropped
    prob_fair   = f_fair.predict_proba(X_scaled[:, fair_cols])[:, 1]
    prob_biased = f_biased.predict_proba(X_scaled)[:, 1]

    combined = d_vals.flatten() * prob_fair + (1 - d_vals.flatten()) * prob_biased
    return combined

def F_predict(X_scaled):
    return (F(X_scaled) >= 0.5).astype(int)

In [ ]:
preds = F_predict(X_test_s)
print("F(x) accuracy:", (preds == y_test).mean().round(3))
# Should match f_fair accuracy within ~1-2%

In [ ]:
import shap

def F_for_shap(X_scaled):
    probs = F(X_scaled)
    return np.column_stack([1 - probs, probs])

explainer = shap.KernelExplainer(F_for_shap, X_train_s[:100])  # use 100-sample background
shap_vals  = explainer.shap_values(X_test_s[:200], nsamples=100)

# Also compute baseline SHAP for comparison
def fair_for_shap(X_scaled):
    p = f_fair.predict_proba(X_scaled[:, fair_cols])[:, 1]
    return np.column_stack([1-p, p])

explainer_base = shap.KernelExplainer(fair_for_shap, X_train_s[:, fair_cols][:100])
shap_base = explainer_base.shap_values(X_test_s[:, fair_cols][:200], nsamples=100)

In [ ]:
import matplotlib.pyplot as plt

feature_names = ['age','priors','juv_fel','juv_misd','juv_other','charge','sex','race']

# Mean absolute SHAP per feature
mean_shap_attack = np.abs(shap_vals[1]).mean(axis=0)
mean_shap_base   = np.abs(shap_base[1]).mean(axis=0)

# For baseline, race was excluded — set it to 0
mean_shap_base_full = np.zeros(8)
mean_shap_base_full[:7] = mean_shap_base

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].barh(feature_names, mean_shap_base_full)
axes[0].set_title("Baseline model — SHAP feature importance")
axes[1].barh(feature_names, mean_shap_attack)
axes[1].set_title("Adversarial model F(x) — SHAP feature importance")
plt.tight_layout()
plt.savefig("shap_comparison.png", dpi=150)

In [ ]:
from sklearn.metrics import confusion_matrix

race_col = X_test[:, 7]
for race_val, race_name in [(0,'African-American'),(1,'Caucasian')]:
    mask = race_col == race_val
    cm = confusion_matrix(y_test[mask], preds[mask])
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn)
    tpr = tp / (tp + fn)
    print(f"{race_name}: FPR={fpr:.3f}, TPR={tpr:.3f}")